In [ ]:
# imports
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# load xml data file
def load_apple_export_data(filepath):

    # read file
    tree = ET.parse(filepath)
    root = tree.getroot()

    # for each record (ie feature)
    records = []

    for record in root.findall('Record'):
        records.append({
            'type': record.get('type'),
            'startDate': record.get('startDate'),
            'endDate': record.get('endDate'),
            'value': record.get('value')
        })

    # convert to dataframe
    df = pd.DataFrame(records)

    # convert startDate and endDate to datetime type
    df['startDate'] = pd.to_datetime(df['startDate'])
    df['endDate'] = pd.to_datetime(df['endDate'])

    return df

# usage
all_health_data = load_apple_export_data('../data/export.xml')

In [ ]:
# find missing dates - FAST
def find_missing_dates(df):
    df = df.copy()

    s = pd.to_datetime(df['startDate']).dt.normalize()
    e = pd.to_datetime(df['endDate']).dt.normalize()

    start = s.min()
    end = e.max()

    # number of days in timeline
    n = (end - start).days + 1

    diff = np.zeros(n + 1, dtype=int)

    start_idx = (s - start).dt.days
    end_idx = (e - start).dt.days

    # mark interval starts and ends
    np.add.at(diff, start_idx, 1)
    np.add.at(diff, end_idx + 1, -1)

    coverage = diff.cumsum()[:-1] > 0

    full_range = pd.date_range(start, end)

    return full_range[~coverage]

# usage
print(find_missing_dates(all_health_data))

DatetimeIndex(['2020-02-24 00:00:00-06:00', '2020-02-25 00:00:00-06:00',
               '2020-02-26 00:00:00-06:00', '2020-02-27 00:00:00-06:00',
               '2020-02-28 00:00:00-06:00', '2020-02-29 00:00:00-06:00',
               '2020-03-01 00:00:00-06:00', '2020-03-02 00:00:00-06:00',
               '2020-03-03 00:00:00-06:00', '2020-03-04 00:00:00-06:00',
               '2020-03-05 00:00:00-06:00', '2020-03-06 00:00:00-06:00',
               '2020-03-07 00:00:00-06:00', '2020-03-08 00:00:00-06:00',
               '2020-03-09 00:00:00-06:00', '2020-03-10 00:00:00-06:00',
               '2020-03-11 00:00:00-06:00', '2020-03-12 00:00:00-06:00',
               '2020-03-13 00:00:00-06:00', '2020-03-14 00:00:00-06:00',
               '2020-03-15 00:00:00-06:00', '2020-03-16 00:00:00-06:00',
               '2020-03-17 00:00:00-06:00', '2020-03-18 00:00:00-06:00',
               '2020-03-19 00:00:00-06:00', '2020-03-20 00:00:00-06:00',
               '2020-03-22 00:00:00-06:00', '2020-0

In [ ]:
# remove large gaps from analysis (ie as seen in test data)
def uninclude_long_gaps(df, gap_length=30):
    df = df.sort_values('startDate').copy()
    df['startDate'] = pd.to_datetime(df['startDate'])

    gaps = df['startDate'].diff().dt.days
    last_gap_idx = gaps[gaps > gap_length].last_valid_index()

    if last_gap_idx is None:
        return df

    # use iloc to handle non-integer indices safely
    pos = df.index.get_loc(last_gap_idx)
    return df.iloc[pos + 1 :]

# usage
all_health_data = uninclude_long_gaps(all_health_data)

In [ ]:
# detect *short-term* spikes in heart rate
def detect_hr_spikes(df, column="HKQuantityTypeIdentifierRestingHeartRate", window=7, threshold=1.5):
    
    rolling_mean = df[column].rolling(window).mean()
    rolling_std = df[column].rolling(window).std()
    
    spikes = df[column] > (rolling_mean + threshold * rolling_std)
    
    df['hr_spike'] = spikes
    
    return df

# usage
all_health_data = detect_hr_spikes(all_health_data)

In [ ]:
# detect long-term deviations in heart rate
# default window time = 30 days
def detect_sustained_shift(
    df,
    column="resting_hr",
    baseline_window=30,
    recent_window=14,
    threshold=3,
    direction="both"  # "up", "down", or "both"
):
    df = df.copy()

    # compute baseline (using past data only)
    df["baseline"] = (
        df[column]
        .rolling(window=baseline_window)
        .mean()
        .shift(1)
    )

    # compute deviation from baseline
    df["deviation"] = df[column] - df["baseline"]

    # identify recent window
    recent = df.tail(recent_window)
    mean_dev = recent["deviation"].mean()

    # 4. determine shift
    if direction == "up":
        shift = mean_dev > threshold
    elif direction == "down":
        shift = mean_dev < -threshold
    else:  # both
        shift = abs(mean_dev) > threshold

    # create summary and return updated df
    summary = {
        "mean_deviation": mean_dev,
        "shift_detected": shift,
        "direction": (
            "up" if mean_dev > threshold else
            "down" if mean_dev < -threshold else
            "none"
        )
    }

    return df, summary

# usage
all_health_data, result = detect_sustained_shift(all_health_data)
print(result)